In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from openai import OpenAI
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)
result.questions

questions=['I just found this course — is it too late to join now?', 'Can I still start the course if I only discovered it recently?', 'If I join late, am I still eligible for a certificate?', 'What’s the deadline if I want to get the course certificate?', 'Do I need to submit the project before submissions close to get certified?']


['I just found this course — is it too late to join now?',
 'Can I still start the course if I only discovered it recently?',
 'If I join late, am I still eligible for a certificate?',
 'What’s the deadline if I want to get the course certificate?',
 'Do I need to submit the project before submissions close to get certified?']

In [12]:
from evaluation_utils import llm_structured

/Users/henrik/Documents/dev/python/llm-zoomcamp/LLM-zoomcamp-04-evaluation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

result.questions
usage.input_tokens, usage.output_tokens

(207, 89)

In [14]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=89, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=296)

In [15]:
from evaluation_utils import calc_total_price_gpt_5_4_mini

In [16]:
total_cost = calc_total_price_gpt_5_4_mini([usage])

In [17]:
total_cost

0.00055575

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course, or can I still sign up?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to finish and submit the project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want the certificate?',
  'document': '74eb249bbf'}]

In [19]:
import pandas as pd

In [20]:
pd.DataFrame(records)

,question,document
0,Can I still join the course if I just found it...,74eb249bbf
1,"Is it too late to start this course, or can I ...",74eb249bbf
2,"If I join the course late, can I still get a c...",74eb249bbf
3,Do I have to finish and submit the project bef...,74eb249bbf
4,What’s the deadline for project submission if ...,74eb249bbf


In [21]:
from evaluation_utils import llm_structured_retry

In [22]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [23]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

100%|██████████| 5/5 [00:09<00:00,  1.99s/it]


In [25]:
cost = calc_total_price_gpt_5_4_mini(usages)
cost

0.003543

In [26]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [27]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

100%|██████████| 103/103 [00:37<00:00,  2.77it/s]


In [29]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [30]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07743225000000002

In [32]:
from evaluation_utils import calc_total_price_gpt_5_4_mini

calc_total_price_gpt_5_4_mini(usages)

0.07743225000000002

In [33]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [35]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)